# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді

**1. Список віршів замість одного суцільного рядка в тренері BPE (2.1)**

`train_from_iterator` отримує кожен вірш як окремий елемент, тож тренер не тримає весь корпус як один текстовий блок під час підрахунку частот пар. Якби ми передали один суцільний рядок (~61 млн символів):
- увесь корпус був би одним величезним елементом: більше пам'яті й вищий ризик OOM;
- зникла б природна одиниця «вірш», яку ми далі використовуємо для пачкового кодування (`encode_batch`) і вставки `<eos>` між віршами: довелося б спершу вручну різати рядок за роздільниками.

**2. Етапи токенізації** (за HF LLM Course, розділ 6.8)

1. **Normalization**: очищення тексту (регістр, зайві пробіли, Unicode-нормалізація, видалення акцентів). У демо нормалізатора немає.
2. **Pre-tokenization**: розбиття тексту на «слова» (за пробілами, пунктуацією). Злиття пар не виходять за межі цих шматків. У демо це `ByteLevel`, який переводить текст у байти.
3. **Model**: алгоритм, що розбиває кожне слово на підслова зі словника (BPE, WordPiece, Unigram). У демо це BPE.
4. **Post-processing**: додавання спеціальних токенів (`[CLS]`, `[SEP]` тощо). У демо відсутній.

Для зворотного перетворення id → текст є ще **decoder** (у демо `ByteLevel`).

**3. `"<unk>"` і `"<eos>"` у 2.1**

- `<unk>` (unknown) замінює символи, яких немає в словнику. У нашому випадку його частка 0,00%, бо `ByteLevel` розкладає будь-який текст на байти.
- `<eos>` (end of sequence) позначає кінець вірша. Ми вставляємо його між віршами в 2.2.

Обидва передані в `special_tokens` тренера, тому отримують перші id у словнику й ніколи не зливаються й не розбиваються.

**4. `VOCAB_SIZE`: 16000 → 5000**

```python
def corpus_stats(tok, texts, batch=1000):
    unk_id = tok.token_to_id("<unk>")
    n_tok = n_unk = 0
    for s in range(0, len(texts), batch):
        for enc in tok.encode_batch(texts[s:s + batch]):
            n_tok += len(enc.ids)
            n_unk += enc.ids.count(unk_id)
    return n_tok, n_unk

texts = poems_df["context_poem"].dropna().tolist()

small_tok = Tokenizer(models.BPE(unk_token="<unk>"))
small_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
small_tok.decoder = decoders.ByteLevel()
small_tok.train_from_iterator(
    texts, trainers.BpeTrainer(vocab_size=5000, special_tokens=SPECIAL_TOKENS)
)

for name, tok in [("16000", bpe_tokenizer), ("5000", small_tok)]:
    n_tok, n_unk = corpus_stats(tok, texts)
    print(f"VOCAB={name}: токенів={n_tok}, на вірш={n_tok / len(texts):.1f}, <unk>={n_unk / n_tok:.2%}")
```

| | VOCAB=16000 | VOCAB=5000 |
|---|---|---|
| Токенів у корпусі | 18 404 050 | 21 598 411 (+17,4%) |
| Токенів на вірш | 1027,1 | 1205,4 |
| Частка `<unk>` | 0,00% | 0,00% |

- **Чому `<unk>` не зростає.** Токенізатор byte-level: усі 256 байтів входять до початкового алфавіту, тож будь-який символ можна закодувати. Менший словник означає лише менше злиттів, і слова розпадаються на більшу кількість коротших підслів.
- **Вплив на довжину.** Той самий текст дає на 17,4% більше токенів. Середній токен коротший: ≈3,3 символа проти ≈2,8 (60 886 196 символів корпусу ÷ кількість токенів).
- **Вплив на `CONTEXT_LEN=64`.** Вікно лишається 64 токени, але покриває ≈180 символів замість ≈212. Модель бачить коротший фрагмент вірша, а за ті самі `MAX_ITERS × BATCH_SIZE × CONTEXT_LEN` токенів проходить менше тексту. Натомість матриці embedding і `lm_head` менші: за розрахунком модель мала б ≈2,1 млн параметрів замість 4,9 млн (модель із таким словником не тренувалась).

**5. Навіщо `<eos>` між віршами (2.2)**

Без `<eos>` кінець одного вірша впритул стикається з початком іншого, і модель бачить один суцільний текст. Наслідки:
- вікна контексту на межі віршів містять непов'язані тексти, а модель вчиться «продовжувати» вірш наступним, зовсім іншим;
- модель не вчиться завершувати твір, і при генерації немає природного сигналу зупинки;
- з `<eos>` модель вчиться передбачати кінець вірша, і його можна використати як стоп-токен.

**6. Формат `train_data` та `val_data` (2.3)**

Це одновимірні тензори `torch.int64` (`torch.Size([16579771])` і `torch.Size([1842197])`). Кожен елемент це **id BPE-токена** (число від 0 до 15999, індекс у словнику). Вони містять увесь корпус підряд, а між віршами стоїть id `<eos>`. `train` це перші 90% послідовності, `val` останні 10%. Це не one-hot і не embedding: вектори утворюються пізніше в `nn.Embedding`.

**7. `assert` у 3.2**

```python
xb, yb = get_batch("train")
print("Форма батчу x:", xb.shape, " y:", yb.shape)
assert (xb[:, 1:] == yb[:, :-1]).all(), "y має бути x, зсунутим на 1 токен"
```

Для кожного вікна `y[t] = x[t+1]`, тобто таргет це вхід, зсунутий на один токен уперед. Тест порівнює `x` без першого стовпця з `y` без останнього. Форма батча `(64, 64)` для `x` і `y`, `assert` пройшов.

**8. Чому `causal_mask` нижньотрикутна (3.3)**

У матриці `scores` `[T, T]` рядок `i` це запит (позиція `i`), стовпець `j` це ключ (позиція `j`). `torch.tril` залишає одиниці там, де `j ≤ i`, тож позиція `i` бачить себе й лише попередні токени. Решту заповнюємо `-inf`, і після `softmax` вага майбутніх токенів дорівнює нулю. Верхньотрикутна маска дала б навпаки: токен бачив би майбутнє й не бачив минулого.

Без маски під час тренування токен на позиції `t` бачив би `x[t+1]`, а це якраз таргет `y[t]`. Модель навчилась би просто копіювати наступний токен, і loss швидко пішов би до нуля. При генерації наступних токенів ще немає, тому такий «навчений» режим не працює, і згенерований текст був би поганої якості.

**9. Найкращий чекпоінт проти останнього (3.10)**

Вони збігаються, якщо `val_loss` знижується до самого кінця (перенавчання немає). Вони відрізняються, якщо `val_loss` досягає мінімуму раніше, а потім росте: модель починає запам'ятовувати train, і найкращим стає чекпоінт із середини тренування.

У моєму запуску (крива 3.9) `val_loss` загалом знижувався до кінця з невеликим шумом: на iter 7999 train 5,562, val 5,646, і найкращий чекпоінт є останнім (iter 7999). Розрив між train і val малий (≈0,08), крива ще не вийшла на плато, тому модель радше недонавчена.

**10. `EVAL_INTERVAL`: 250 → 50**

```python
EVAL_INTERVAL = 50    # було 250
```

| | `EVAL_INTERVAL=250` | `EVAL_INTERVAL=50` |
|---|---|---|
| Точок на кривій | 33 | 161 |
| Час тренування | 6,6 хв | 10,1 хв (+3,5 хв, ≈ +53%) |
| Фінальний val_loss | 5,639 | 5,646 |

Крива не стала гладкішою: сусідні точки коливаються на ≈0,02-0,05, бо `estimate_loss()` усереднює лише `EVAL_ITERS=50` батчів. Вона стала детальнішою на початку (падіння з 9,76 до 7,46 за перші 100 ітерацій раніше ховалось між iter 0 і 250), але висновки не змінились. Частіші оцінки лише додають вартості, а тренування від них не залежить. Різниця фінальних val_loss це випадковість. Робити інтервал ще меншим немає сенсу: час росте лінійно, а крива лише шумнішає. Для гладшої кривої краще збільшити `EVAL_ITERS`.

**11. Математика `temperature` (4.2)**

Перед `softmax` логіти ділять на температуру `T`:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Відношення ймовірностей двох токенів дорівнює $e^{(z_i - z_j)/T}$.
- При `T < 1` різниці логітів збільшуються, і розподіл гострішає. Коли `T → 0`, це argmax (жадібний вибір).
- При `T > 1` різниці зменшуються, і розподіл вирівнюється. Коли `T → ∞`, він рівномірний (`1/V`).

Приклад для логітів `[4, 2, 0]`: при `T=1` ймовірності ≈ 0,87 / 0,12 / 0,02, при `T=0.2` ≈ 1,00 / 0,00005 / ~0, при `T=1.4` ≈ 0,77 / 0,18 / 0,04.

При дуже високій `T` сумарна маса «хвоста» з тисяч малоймовірних токенів стає значною, і вибір стає майже випадковим, тому текст втрачає зв'язність. У моєму запуску при `T=1.4` текст лишався зв'язним значною мірою завдяки `top_p=0.8`, який відсікає хвіст.

**12. `top_p`**

`top_p` (nucleus sampling) сортує токени за ймовірністю, лишає найменшу множину, сума ймовірностей якої ≥ `p`, перенормовує її й семплує лише з неї. На відміну від `top_k`, розмір множини адаптивний: якщо модель впевнена, лишається 1-2 токени, якщо ні, то більше. Тому при `top_p=0.1` вибір майже жадібний, а при `0.8` ширший. У моєму запуску початки текстів при `(0.2, 0.1)` і `(0.8, 0.6)` майже збіглися, значною мірою через малий `top_p`.

**13. `enable_thinking=True` у 4.2**

```python
text = hf_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
)
```

Модель спочатку генерує блок `<think>…` із міркуваннями (англійською), і тільки потім відповідь. Усі 300 токенів (`max_new_tokens`) пішли на міркування, тому **сама відповідь не з'явилась**, а вивід обірвався на пів слові. У всіх трьох запусках міркування починаються однаково («Okay, the user wants me to continue the sentence…»), і модель плутає університет (Харківська, Хмельницька політехніка), хоча в промпті Житомирська. Між температурами видно лише дрібні відмінності у формулюваннях, а результат однаковий: до відповіді справа не дійшла. Щоб дістатися відповіді, довелося б збільшити `max_new_tokens` (я цього не перевіряв). Для цього завдання thinking лише витрачає токени, тому в демо повернув `enable_thinking=False`.

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

### Обраний датасет

Для роботи використовую датасет **[Yehor/news-headlines-ubercorpus](https://huggingface.co/datasets/Yehor/news-headlines-ubercorpus)** з Hugging Face Datasets: новинні заголовки українською мовою.

- **Жанр:** заголовки новин (короткі, однорядкові тексти, інший регістр, ніж поезія з демо).
- **Обсяг:** 3 976 006 заголовків, колонка з текстом `headline`.
- **Вибірка:** випадкові 300 000 заголовків (`shuffle(seed=42)`), без дублікатів.
- **Ліцензія:** CC BY-NC-SA 4.0 (некомерційне використання).

In [8]:
from datasets import load_dataset
import pandas as pd

N_ROWS = 300000

ds = load_dataset("Yehor/news-headlines-ubercorpus", split="train")
print("Усього заголовків у датасеті:", len(ds))

news_df = ds.shuffle(seed=42).select(range(N_ROWS)).to_pandas()
news_df = news_df.drop_duplicates("headline")

print(len(news_df), news_df.columns.tolist())
print(news_df.head(10).to_string())

Усього заголовків у датасеті: 3976006
300000 ['headline', 'length']
                                                                                                           headline  length
0                                               Порошенко вирішив надати гуманітарну допомогу Ємену, Судану і Конго      67
1                                                     Прес-конференція до Всесвітнього дня боротьби з туберкульозом      61
2                                                           Лівійські повстанці ведуть наступ у декількох напрямках      55
3                                                  У рамках місії НАТО Польща відправляє до Литви чотири винищувачі      64
4                                                                            Брат Балоги просить захисту в опозиції      38
5  Делегації оборонних відомств України та Республіки Хорватія на чолі із Міністрами оборони побували у Севастополі     112
6                                                               

In [9]:
corpus_text = "\n".join(news_df["headline"].dropna().tolist())

print("Розмір корпусу:", len(corpus_text), "символів")
print("Перші 500 символів корпусу:\n", corpus_text[:500])

Розмір корпусу: 18737916 символів
Перші 500 символів корпусу:
 Порошенко вирішив надати гуманітарну допомогу Ємену, Судану і Конго
Прес-конференція до Всесвітнього дня боротьби з туберкульозом
Лівійські повстанці ведуть наступ у декількох напрямках
У рамках місії НАТО Польща відправляє до Литви чотири винищувачі
Брат Балоги просить захисту в опозиції
Делегації оборонних відомств України та Республіки Хорватія на чолі із Міністрами оборони побували у Севастополі
У Каневі зросте тариф на воду
Виїзний брифінг на огляд роботи сучасного рентген-кабінету із цифро


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [10]:
from transformers import AutoTokenizer

sample_texts = news_df["headline"].dropna().tolist()[:20000]

old_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
new_tok = old_tok.train_new_from_iterator(sample_texts, vocab_size=5000)

example = "Порошенко вирішив надати гуманітарну допомогу Ємену, Судану і Конго"
print("Qwen (старий):", len(old_tok.tokenize(example)), "токенів")
print("Новий:        ", len(new_tok.tokenize(example)), "токенів")
print(new_tok.tokenize(example))

Qwen (старий): 33 токенів
Новий:         17 токенів
['ÐŁÐ¾ÑĢÐ¾ÑĪÐµÐ½ÐºÐ¾', 'ĠÐ²Ð¸ÑĢÑĸÑĪÐ¸Ð²', 'ĠÐ½Ð°', 'Ð´Ð°ÑĤÐ¸', 'ĠÐ³ÑĥÐ¼', 'Ð°Ð½ÑĸÑĤÐ°ÑĢ', 'Ð½Ñĥ', 'ĠÐ´Ð¾Ð¿Ð¾Ð¼Ð¾Ð³Ñĥ', 'ĠÐĦ', 'Ð¼ÐµÐ½', 'Ñĥ', ',', 'ĠÐ¡ÑĥÐ´', 'Ð°Ð½Ñĥ', 'ĠÑĸ', 'ĠÐļÐ¾Ð½', 'Ð³Ð¾']


In [11]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

VOCAB = 500
UNK = "[UNK]"

bpe = Tokenizer(models.BPE(unk_token=UNK))
bpe.pre_tokenizer = pre_tokenizers.Whitespace()
bpe.train_from_iterator(sample_texts, trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=[UNK]))

wp = Tokenizer(models.WordPiece(unk_token=UNK))
wp.pre_tokenizer = pre_tokenizers.Whitespace()
wp.train_from_iterator(sample_texts, trainers.WordPieceTrainer(vocab_size=VOCAB, special_tokens=[UNK]))

uni = Tokenizer(models.Unigram())
uni.pre_tokenizer = pre_tokenizers.Whitespace()
uni.train_from_iterator(
    sample_texts, trainers.UnigramTrainer(vocab_size=VOCAB, special_tokens=[UNK], unk_token=UNK)
)

held_out = news_df["headline"].dropna().tolist()[20000:21000]
words = ["України", "кіберзлочинців"]

rows = []
for name, tok in {"BPE": bpe, "WordPiece": wp, "Unigram": uni}.items():
    row = {
        "алгоритм": name,
        "словник": tok.get_vocab_size(),
        "токенів на 1000 заголовків": sum(len(tok.encode(t).tokens) for t in held_out),
    }
    for w in words:
        row[w] = tok.encode(w).tokens
    rows.append(row)

results_df = pd.DataFrame(rows)
display(results_df)

,алгоритм,словник,токенів на 1000 заголовків,України,кіберзлочинців
0,BPE,500,29332,[України],"[кі, б, ер, з, ло, чи, н, ців]"
1,WordPiece,500,34514,[України],"[к, ##і, ##б, ##ер, ##з, ##ло, ##чи, ##н, ##ці..."
2,Unigram,500,30620,"[Україн, и]","[к, і, б, ер, з, ло, чи, н, ці, в]"


### Висновок за порівнянням токенізаторів (vocab_size = 500)

| Алгоритм | Словник | Токенів на 1000 заголовків | «України» | «кіберзлочинців» |
|---|---|---|---|---|
| BPE | 500 | 29 332 | 1 токен | 8 токенів |
| WordPiece | 500 | 34 514 (+17,7%) | 1 токен | 10 токенів |
| Unigram | 500 | 30 620 (+4,4%) | 2 токени | 10 токенів |

**Найкоротші послідовності дав BPE** (29 332 токени на 1000 заголовків, які не входили до навчального зразка), далі Unigram, а найдовші у WordPiece. Частотне слово «України» BPE і WordPiece зберігають як один токен, а Unigram розбиває на два (`Україн` + `и`). Рідкісне «кіберзлочинців» усі три розбивають на 8-10 шматків.

**Чи узгоджується це з туторіалом:**
- BPE на кожному кроці зливає найчастішу пару, тож найкраще стискає тексти, схожі на навчальні, і дає найменше токенів.
- WordPiece обирає пари за балом, який зменшується, якщо частини пари самі часті. Він не збігається зі стисненням за частотою, тому за малого словника дає довші послідовності. Також `##`-префікси створюють окремі варіанти літер усередині слова, що забирає частину з 500 місць у словнику (це моє пояснення, окремо не перевірялось).
- Unigram стартує з великого словника й вилучає токени за втратою правдоподібності, а токенізує слово найімовірнішою сегментацією. Тому результат близький до BPE, але не збігається з ним за межами злиття (наприклад, поділ «Україн» + «и»).

**Обмеження:** порівняння зроблено на одному зразку (20 000 заголовків для навчання, 1000 для оцінки), з розбиттям за пробілами (`Whitespace`) і лише при `vocab_size=500`. Для більшого словника різниця між алгоритмами могла б зменшитися.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

### Пункт 3.1

In [1]:
import os, sys, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Використовуємо:", device)

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = "/content/drive/MyDrive/ai_systems/lab3"
else:
    ARTIFACT_DIR = "lab3"

MODELS_DIR = os.path.join(ARTIFACT_DIR, "models")
RUNS_DIR = os.path.join(ARTIFACT_DIR, "runs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print("Артефакти:", ARTIFACT_DIR)

Використовуємо: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Артефакти: /content/drive/MyDrive/ai_systems/lab3


In [13]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 8000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

gpt_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
gpt_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
gpt_tokenizer.decoder = decoders.ByteLevel()

headlines = news_df["headline"].dropna().tolist()
gpt_tokenizer.train_from_iterator(
    headlines, trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
)

stoi = gpt_tokenizer.get_vocab()
itos = [None] * len(stoi)
for tok, idx in stoi.items():
    itos[idx] = tok
UNK_ID, EOS_ID = stoi["<unk>"], stoi["<eos>"]

def tokenize(text: str) -> list[str]:
    return gpt_tokenizer.encode(text).tokens

def encode(tokens: list[str]) -> list[int]:
    return [stoi.get(t, UNK_ID) for t in tokens]

def decode(ids: list[int]) -> str:
    return gpt_tokenizer.decode(ids)

print("Розмір словника:", len(itos))
print(decode(gpt_tokenizer.encode(headlines[0]).ids))

Розмір словника: 8000
Порошенко вирішив надати гуманітарну допомогу Ємену, Судану і Конго


In [14]:
encodings = gpt_tokenizer.encode_batch(headlines)
token_ids = np.concatenate(
    [np.array(e.ids + [EOS_ID], dtype=np.int32) for e in encodings]
)
del encodings

print(f"Токенів у корпусі: {len(token_ids)}")
print(f"Токенів поза словником (<unk>): {(token_ids == UNK_ID).mean():.1%}")

data = torch.from_numpy(token_ids.astype(np.int64))
split_idx = int(len(data) * 0.9)
train_data, val_data = data[:split_idx], data[split_idx:]
print("train:", train_data.shape, " val:", val_data.shape)

Токенів у корпусі: 5109169
Токенів поза словником (<unk>): 0.0%
train: torch.Size([4598252])  val: torch.Size([510917])


### Пункт 3.2

In [15]:
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
EMBED_DIM = 128
DROPOUT = 0.1
MAX_ITERS = 8000
EVAL_INTERVAL = 250
EVAL_ITERS = 50

def get_batch(split: str, context_len: int):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - context_len - 1, (BATCH_SIZE,))
    x = torch.stack([source[i:i + context_len] for i in ix])
    y = torch.stack([source[i + 1:i + context_len + 1] for i in ix])
    return x.to(device), y.to(device)

In [16]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim: int, head_dim: int, context_len: int, dropout: float):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        return weights @ v

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([
            AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)
        ])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [18]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)




class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


In [19]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, n_heads: int, n_layers: int,
                 context_len: int, dropout: float):
        super().__init__()
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0,
                 top_k: int | None = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

In [20]:
from torch.utils.tensorboard import SummaryWriter

@torch.no_grad()
def estimate_loss(model, context_len):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for i in range(EVAL_ITERS):
            xb, yb = get_batch(split, context_len)
            _, loss = model(xb, yb)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train_variant(name, context_len, n_layers, n_heads):
    run_name = f"{name}_ctx{context_len}_L{n_layers}_H{n_heads}"
    model = MiniGPT(len(itos), EMBED_DIM, n_heads, n_layers, context_len, DROPOUT).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, run_name))
    ckpt_path = os.path.join(MODELS_DIR, f"minigpt_{name}.pt")

    best_val, best_iter = float("inf"), None
    start = time.time()
    for it in range(MAX_ITERS):
        xb, yb = get_batch("train", context_len)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            losses = estimate_loss(model, context_len)
            writer.add_scalar("Loss/train", losses["train"], it)
            writer.add_scalar("Loss/val", losses["val"], it)
            if losses["val"] < best_val:
                best_val, best_iter = losses["val"], it
                torch.save(model.state_dict(), ckpt_path)
    writer.close()

    minutes = (time.time() - start) / 60
    print(f"{run_name}: params={n_params:,}  best_val_loss={best_val:.3f} (iter {best_iter})  {minutes:.1f} хв")
    return dict(variant=name, context_len=context_len, n_layers=n_layers, n_heads=n_heads,
                params=n_params, best_val_loss=best_val, best_iter=best_iter, minutes=minutes)

In [21]:
import pandas as pd

VARIANTS = {
    "v1": dict(context_len=64,  n_layers=4, n_heads=4),
    "v2": dict(context_len=64,  n_layers=8, n_heads=4),
    "v3": dict(context_len=128, n_layers=4, n_heads=4),
}

results = [train_variant(name, **cfg) for name, cfg in VARIANTS.items()]

results_gpt_df = pd.DataFrame(results)
results_gpt_df.to_csv(os.path.join(ARTIFACT_DIR, "results.csv"), index=False)
display(results_gpt_df)

v1_ctx64_L4_H4: params=2,856,000  best_val_loss=4.736 (iter 7999)  4.9 хв
v2_ctx64_L8_H4: params=3,647,552  best_val_loss=4.568 (iter 7999)  7.6 хв
v3_ctx128_L4_H4: params=2,864,192  best_val_loss=4.709 (iter 7999)  9.2 хв


,variant,context_len,n_layers,n_heads,params,best_val_loss,best_iter,minutes
0,v1,64,4,4,2856000,4.735689,7999,4.934844
1,v2,64,8,4,3647552,4.568037,7999,7.580220
2,v3,128,4,4,2864192,4.709235,7999,9.176928


### Пункт 3.3

In [23]:
PROMPT = "У Житомирі"

def generate_text(name, cfg, prompt, max_new_tokens=40, temperature=0.8, top_k=40):
    model = MiniGPT(len(itos), EMBED_DIM, cfg["n_heads"], cfg["n_layers"],
                    cfg["context_len"], DROPOUT).to(device)
    model.load_state_dict(torch.load(os.path.join(MODELS_DIR, f"minigpt_{name}.pt"),
                                     map_location=device))
    prompt_ids = torch.tensor([gpt_tokenizer.encode(prompt).ids], dtype=torch.long).to(device)
    out_ids = model.generate(prompt_ids, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out_ids[0].tolist())

for name, cfg in VARIANTS.items():
    torch.manual_seed(0)
    print(f"=== {name} {cfg} ===")
    print(generate_text(name, cfg, PROMPT), "\n")

=== v1 {'context_len': 64, 'n_layers': 4, 'n_heads': 4} ===
У Житомирі, що не побний, до 3У США виступ"Чорст за фактомку у вбивстві" на у Китаїстльться на 2"ЗЕУ Львові: На 

=== v2 {'context_len': 64, 'n_layers': 8, 'n_heads': 4} ===
У Житомирі затримано трьох людей по службувалиши до смертіВиробництво Держдепартаменту в Києві будуть випускати на коортБразильця і Казбельна, який с 

=== v3 {'context_len': 128, 'n_layers': 4, 'n_heads': 4} ===
У Житомирі, що витримався за газ до судуВиробляти українські військові навчання на коронавірусУ Києві завтра виявили незаконний у зрадіП'яч: в Україні знову знову мають по 



### Висновки (3.3)

Промпт для всіх варіантів однаковий («У Житомирі»), `temperature=0.8`, `top_k=40`, однакове зерно (`torch.manual_seed(0)`). Генерація на чекпоінті з найкращим `val_loss` (в усіх трьох це остання ітерація, 7999). Заголовки в виводі «склеєні» («смертіВиробництво», «коронавірусУ Києві»), бо `decode()` ховає спецтокен `<eos>`, яким модель позначає межу заголовків.

| Варіант | Що змінено | Параметри | Найкращий val_loss | Perplexity (e^loss) |
|---|---|---|---|---|
| v1 | база: `CONTEXT_LEN=64`, 4 шари | 2,86 млн | 4,736 | ≈114 |
| v2 | 8 шарів | 3,65 млн | **4,568** | ≈96 |
| v3 | `CONTEXT_LEN=128` | 2,86 млн | 4,709 | ≈111 |

**Який варіант кращий за `val_loss`:** v2 (8 шарів), різниця з базою 0,168 (≈3,5%).

**Чи помітно це за текстом:** так, але помірно.
- **v1** дає найбільше зламаних слів («побний», «Чорст», «фактомку», «Китаїстльться») і майже не тримає зв'язку з промптом.
- **v2** найкраще продовжує промпт: «У Житомирі затримано трьох людей» є граматично правильним новинним початком. Далі з'являються вигадані слова («службувалиши», «коортБразильця»), але є й впізнавані фрази («Держдепартаменту в Києві будуть випускати»).
- **v3** будує довші відрізки з реальних слів («що витримався за газ до суду», «військові навчання»), але речення беззмістовні, а в кінці є повтор «знову знову».

Усі три тексти далекі від зв'язних, що очікувано для моделей на 3-4 млн параметрів. Це один зразок з одним зерном, тому по ньому не можна судити строго: головний показник це `val_loss`.

**Що дало найбільший ефект:** збільшення глибини (4 → 8 шарів): −0,168. Подвоєння контексту дало лише −0,027, ймовірно тому, що заголовки короткі (≈17 токенів), і вікно 64 токени вже містить кілька заголовків. Вплив розміру словника не перевірявся: у всіх варіантах `VOCAB_SIZE=8000`.

**Обмеження:** по одному запуску на варіант; в усіх варіантів найкращим став останній чекпоінт, а крива ще падала, тобто моделі недонавчені, і довше тренування змінило б різницю. v2 також має більше параметрів (3,65 проти 2,86 млн), тож його перевага змішана з розміром моделі. Час тренування: v1 4,9 хв, v2 7,6 хв, v3 9,2 хв.

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [24]:
import gc, torch

del ds, news_df, headlines, corpus_text, token_ids, data, train_data, val_data
del old_tok, new_tok, bpe, wp, uni, sample_texts, held_out
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: зайнято 0.02 GB


### 4.0. Запустіть приклади виконання різних NLP задач

In [2]:
!pip install "transformers[sentencepiece]==5.18.0"

In [3]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989}]
[{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [4]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.844595193862915, 0.11197695881128311, 0.04342786595225334]}

In [5]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to use a keyboard, to use a keyboard and to use a keyboard, to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and to use a keyboard and t

In [6]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619743525981903,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052695631980896,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [7]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [8]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [9]:
import gc
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="tabularisai/multilingual-sentiment-analysis")
print(classifier([
    "Я чекав на цей курс усе життя!",
    "Сервіс жахливий, більше сюди не прийду.",
    "Доставку привезли вчасно, все працює нормально.",
]))
del classifier; gc.collect()

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'Positive', 'score': 0.6798710227012634}, {'label': 'Negative', 'score': 0.5587374567985535}, {'label': 'Positive', 'score': 0.8768420815467834}]


2350

In [10]:
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
)
print(classifier(
    "Цей курс присвячений бібліотеці Transformers",
    candidate_labels=["освіта", "політика", "бізнес"],
))
del classifier; gc.collect()

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

{'sequence': 'Цей курс присвячений бібліотеці Transformers', 'labels': ['освіта', 'політика', 'бізнес'], 'scores': [0.957226574420929, 0.021397173404693604, 0.021376287564635277]}


270

In [11]:
generator = pipeline("text-generation", model="malteos/gpt2-uk")
print(generator(
    "Одного ранку студент прокинувся від дивного шуму",
    max_new_tokens=40, num_return_sequences=2, do_sample=True,
))
del generator; gc.collect()

config.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  511MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: malteos/gpt2-uk
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json:   0%|          | 0.00/3.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Одного ранку студент прокинувся від дивного шуму, і хлопець почав шукати його:\n— Ти думаєш, він їде на гору, а я… Я знаю його. Але він не каже, що в мене нічого немає, він може втекти.'}, {'generated_text': 'Одного ранку студент прокинувся від дивного шуму й почав:\n— Вибух на Чорнобильській АЕС,— відповів хлопець.— Я зрозумів, що в нас немає часу, щоб вилізти з тунелю. Я не знаю, що відбувається'}]


2526

In [12]:
unmasker = pipeline("fill-mask", model="xlm-roberta-base")
print(unmasker("Цей курс навчить вас усього про <mask> моделі.", top_k=2))
del unmasker; gc.collect()

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[{'score': 0.11329878866672516, 'token': 44817, 'token_str': 'створення', 'sequence': 'Цей курс навчить вас усього про створення моделі.'}, {'score': 0.08669208735227585, 'token': 6317, 'token_str': 'ці', 'sequence': 'Цей курс навчить вас усього про ці моделі.'}]


324

In [13]:
ner = pipeline(
    "ner",
    model="EvanD/xlm-roberta-base-ukrainian-ner-ukrner",
    aggregation_strategy="simple",
)
print(ner("Мене звати Тарас, я працюю в компанії Hugging Face у Києві."))
del ner; gc.collect()

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

[{'entity_group': 'PER', 'score': np.float32(0.9935622), 'word': 'Тарас', 'start': 11, 'end': 16}, {'entity_group': 'ORG', 'score': np.float32(0.9084532), 'word': 'Hugging Face', 'start': 38, 'end': 50}, {'entity_group': 'LOC', 'score': np.float32(0.9937795), 'word': 'Києві', 'start': 53, 'end': 58}]


271

In [14]:
table = {
    "Місто": ["Київ", "Львів", "Житомир"],
    "Населення": ["2900000", "720000", "260000"],
    "Область": ["Київська", "Львівська", "Житомирська"],
}
tqa = pipeline("table-question-answering")
print(tqa(query="Яке населення у Львова?", table=table))
print(tqa(query="What is the population of Львів?", table=table))
del tqa; gc.collect()

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{'answer': 'AVERAGE > 2900000', 'coordinates': [(0, 1)], 'cells': ['2900000'], 'aggregator': 'AVERAGE'}
{'answer': 'AVERAGE > 720000', 'coordinates': [(1, 1)], 'cells': ['720000'], 'aggregator': 'AVERAGE'}


274

### Коментар до 4.1

Для кожної задачі підібрано модель зі знанням української:

| Задача | Модель |
|---|---|
| sentiment-analysis | `tabularisai/multilingual-sentiment-analysis` |
| zero-shot-classification | `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` |
| text-generation | `malteos/gpt2-uk` |
| fill-mask | `xlm-roberta-base` |
| NER | `EvanD/xlm-roberta-base-ukrainian-ner-ukrner` |
| table-question-answering | `google/tapas-base-finetuned-wtq` (за замовчуванням) |

- **Zero-shot:** тема «Цей курс присвячений бібліотеці Transformers» класифікована як «освіта» зі score ≈ 0,96.
- **Text-generation:** `gpt2-uk` продовжує український промпт зв'язними реченнями.
- **Fill-mask:** найімовірніше слово «створення» (score ≈ 0,11) дає правильне словосполучення.
- **NER:** сутність «Тарас» розпізнано як `PER` зі score ≈ 0,99.
- **Table QA:** TAPAS англомовний. На українському запиті «Яке населення у Львова?» модель обрала клітинку з рядка Києва (2900000), тобто помилилась. На англійському запиті «What is the population of Львів?» вона обрала правильну клітинку (720000). Отже, для цієї задачі модель з підтримкою української не знайдено, і для українських запитів потрібна окрема модель або переклад запиту.

---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

### 5.1. Вибір задачі та метрики

Для порівняння я обрав задачу **sentiment-analysis**, бо вона бінарна, а правильні мітки легко підготувати вручну (20 власних речень: 10 позитивних і 10 негативних). Метрики: **Accuracy** і **F1** з бібліотеки `evaluate`. Порівнюю дві моделі з розділу 4, які підтримують українську й працюють за різним принципом: `tabularisai/multilingual-sentiment-analysis` (спеціалізований класифікатор тональності) і `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` (zero-shot-класифікація з мітками «позитивний» / «негативний»).

### Пунтк 5.1


In [15]:
!pip install -q evaluate

import time
import evaluate
import pandas as pd
from transformers import pipeline

texts = [
    # позитивні (1)
    "Чудовий фільм, дивилися всією родиною з величезним задоволенням.",
    "Доставка прийшла раніше, ніж обіцяли, а товар виявився навіть кращим за очікування.",
    "Дуже привітний персонал, смачна кава й затишна атмосфера.",
    "Нарешті вийшло оновлення, яке реально прискорило роботу програми.",
    "Лекція була цікавою, викладач пояснив усе зрозуміло й на прикладах.",
    "Купив цей телефон місяць тому і не нарікаю: батарея тримає два дні.",
    "Дякую команді за швидку допомогу, проблему вирішили за п'ять хвилин!",
    "Це найкраща подорож у моєму житті, обов'язково поїду ще раз.",
    "Книга захопила з першої сторінки, не міг відірватися до ранку.",
    "Ціни приємно здивували, а якість на висоті.",
    # негативні (0)
    "Жахливий сервіс: чекали замовлення годину, і воно прийшло холодним.",
    "Програма постійно вилітає, користуватися нею неможливо.",
    "Дуже розчарований покупкою, товар зламався вже на другий день.",
    "Фільм нудний і затягнутий, пів залу пішло ще до фіналу.",
    "Служба підтримки не відповідає, а гроші мені так і не повернули.",
    "У номері було брудно, а персонал поводився грубо.",
    "Лекція видалася безглуздою, нічого корисного я не дізнався.",
    "Не раджу це місце: дорого, шумно й неприємний запах.",
    "Квитки подорожчали вдвічі, а умови поїздки погіршилися.",
    "Ніколи більше не замовлятиму тут, втратив і час, і гроші.",
]
labels = [1] * 10 + [0] * 10   # 1 = позитивний, 0 = негативний

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def score(preds):
    return (accuracy.compute(predictions=preds, references=labels)["accuracy"],
            f1.compute(predictions=preds, references=labels)["f1"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00


In [16]:
clf = pipeline("sentiment-analysis", model="tabularisai/multilingual-sentiment-analysis")
clf(texts[:1])                               # прогрів, щоб не рахувати перший виклик

t0 = time.time()
out_a = clf(texts)
time_a = time.time() - t0

print("Мітки моделі A:", sorted({o["label"] for o in out_a}))
preds_a = [1 if "positive" in o["label"].lower() else 0 for o in out_a]
acc_a, f1_a = score(preds_a)
params_a = clf.model.num_parameters() / 1e6
print(f"A: accuracy={acc_a:.2f}  f1={f1_a:.2f}  час={time_a:.2f} с  параметри={params_a:.0f} млн")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Мітки моделі A: ['Negative', 'Neutral', 'Positive', 'Very Negative', 'Very Positive']
A: accuracy=0.85  f1=0.82  час=0.18 с  параметри=135 млн


In [17]:
zs = pipeline("zero-shot-classification",
              model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")
LABELS = ["позитивний", "негативний"]
zs(texts[:1], candidate_labels=LABELS)       # прогрів

t0 = time.time()
out_b = zs(texts, candidate_labels=LABELS)
time_b = time.time() - t0

preds_b = [1 if o["labels"][0] == "позитивний" else 0 for o in out_b]
acc_b, f1_b = score(preds_b)
params_b = zs.model.num_parameters() / 1e6
print(f"B: accuracy={acc_b:.2f}  f1={f1_b:.2f}  час={time_b:.2f} с  параметри={params_b:.0f} млн")

pd.DataFrame([
    {"модель": "tabularisai/multilingual-sentiment-analysis", "accuracy": acc_a, "f1": f1_a,
     "час, с": round(time_a, 2), "параметри, млн": round(params_a)},
    {"модель": "mDeBERTa zero-shot", "accuracy": acc_b, "f1": f1_b,
     "час, с": round(time_b, 2), "параметри, млн": round(params_b)},
])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

B: accuracy=1.00  f1=1.00  час=1.99 с  параметри=279 млн


,модель,accuracy,f1,"час, с","параметри, млн"
0,tabularisai/multilingual-sentiment-analysis,0.85,0.823529,0.18,135
1,mDeBERTa zero-shot,1.00,1.000000,1.99,279


### 5.2. Порівняння моделей

| Модель | Accuracy | F1 | Час на 20 речень | Параметри |
|---|---|---|---|---|
| `tabularisai/multilingual-sentiment-analysis` | 0,85 | 0,82 | 0,18 с | 135 млн |
| `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` (zero-shot) | **1,00** | **1,00** | 1,99 с | 279 млн |

**Точність:** zero-shot-модель правильно класифікувала всі 20 речень. Спеціалізований класифікатор помилився в 3 реченнях, і всі помилки це позитивні речення, віднесені до не-позитивних (F1 = 0,82 при 17/20 правильних, хибнопозитивних немає). Ймовірна причина: модель має п'ять міток (`Very Negative … Very Positive`, зокрема `Neutral`), а для бінарної оцінки я зводив усе, що не містить «positive», до класу «негативний».

**Швидкість і розмір:** модель A у ~11 разів швидша (0,18 проти 1,99 с) і вдвічі менша за кількістю параметрів. Zero-shot повільніший, бо для кожного речення NLI-модель запускається окремо для кожної мітки.

**Обмеження:** вибірка лише з 20 речень, тому одна помилка змінює accuracy на 5 пунктів. Різниця між моделями може бути частково шумом.

### Пунтк 5.3

In [18]:
BASE_LABELS = ["позитивний", "негативний"]

experiments = [
    ("база",                            dict(candidate_labels=BASE_LABELS), "позитивний"),
    ("шаблон гіпотези українською",     dict(candidate_labels=BASE_LABELS, hypothesis_template="Цей відгук {}."), "позитивний"),
    ("мітки «добре» / «погано»",        dict(candidate_labels=["добре", "погано"]), "добре"),
    ("англійські мітки",                dict(candidate_labels=["positive", "negative"]), "positive"),
]

rows = []
for name, kwargs, pos in experiments:
    t0 = time.time()
    out = zs(texts, **kwargs)
    dt = time.time() - t0
    preds = [1 if o["labels"][0] == pos else 0 for o in out]
    a, f = score(preds)
    rows.append({"налаштування": name, "accuracy": a, "f1": f, "час, с": round(dt, 2)})

# поріг впевненості: «позитивний» лише якщо score >= 0.95 (без повторного запуску моделі)
THRESHOLD = 0.95
preds = [1 if (o["labels"][0] == "позитивний" and o["scores"][0] >= THRESHOLD) else 0 for o in out_b]
a, f = score(preds)
rows.append({"налаштування": f"поріг впевненості {THRESHOLD}", "accuracy": a, "f1": f, "час, с": None})

results_5_df = pd.DataFrame(rows)
display(results_5_df)

,налаштування,accuracy,f1,"час, с"
0,база,1.00,1.000000,1.18
1,шаблон гіпотези українською,1.00,1.000000,1.03
2,мітки «добре» / «погано»,1.00,1.000000,1.00
3,англійські мітки,1.00,1.000000,1.00
4,поріг впевненості 0.95,0.85,0.823529,NaN


### 5.3. Експерименти з налаштуваннями (zero-shot, кращою моделлю обрано mDeBERTa)

Змінювався один параметр за раз відносно бази.

| Налаштування | Accuracy | F1 |
|---|---|---|
| База (мітки «позитивний» / «негативний») | 1,00 | 1,00 |
| Шаблон гіпотези українською (`"Цей відгук {}."`) | 1,00 | 1,00 |
| Мітки «добре» / «погано» | 1,00 | 1,00 |
| Англійські мітки (`positive` / `negative`) | 1,00 | 1,00 |
| Поріг впевненості 0,95 | 0,85 | 0,82 |

**Найкращий результат:** mDeBERTa zero-shot з будь-якою з чотирьох комбінацій міток/шаблону (accuracy = F1 = 1,00).

**Вплив налаштувань проти вибору моделі:** вибір моделі дав більший ефект. Перехід від `tabularisai/multilingual-sentiment-analysis` до mDeBERTa підняв accuracy з 0,85 до 1,00 (+0,15). Налаштування мітки або шаблону не змінили нічого, а поріг 0,95 лише погіршив результат: три правильні позитивні відповіді мали впевненість нижче за 0,95 і були відкинуті як «негативні».

**Обмеження:** база вже дає 1,00, тож налаштування не мали що покращувати (ефект стелі). На 20 реченнях невелика різниця між варіантами не показова, а час інференсу між запусками коливається (1,0-2,0 с).

In [21]:
rows = []
for r in results_gpt_df.itertuples():
    rows.append({
        "розділ": "3 mini-GPT",
        "модель/налаштування": f"{r.variant} (ctx={r.context_len}, layers={r.n_layers}, heads={r.n_heads})",
        "метрика": "val_loss", "значення": round(r.best_val_loss, 4),
        "параметри, млн": round(r.params / 1e6, 2), "час, с": round(r.minutes * 60, 1),
    })

for name, a, f, t, p in [
    ("tabularisai/multilingual-sentiment-analysis", acc_a, f1_a, time_a, params_a),
    ("mDeBERTa zero-shot", acc_b, f1_b, time_b, params_b),
]:
    for metric, val in (("accuracy", a), ("f1", f)):
        rows.append({"розділ": "5.2 порівняння моделей", "модель/налаштування": name,
                     "метрика": metric, "значення": round(val, 4),
                     "параметри, млн": round(p), "час, с": round(t, 2)})

for r in results_5_df.itertuples():
    for metric, val in (("accuracy", r.accuracy), ("f1", r.f1)):
        rows.append({"розділ": "5.3 налаштування mDeBERTa", "модель/налаштування": r.налаштування,
                     "метрика": metric, "значення": round(val, 4),
                     "параметри, млн": None, "час, с": getattr(r, "_4")})

results_all_df = pd.DataFrame(rows)
results_all_df.to_csv(os.path.join(ARTIFACT_DIR, "results.csv"), index=False)
display(results_all_df)

,розділ,модель/налаштування,метрика,значення,"параметри, млн","час, с"
0,3 mini-GPT,"v1 (ctx=64, layers=4, heads=4)",val_loss,4.7357,2.86,296.10
1,3 mini-GPT,"v2 (ctx=64, layers=8, heads=4)",val_loss,4.5680,3.65,454.80
2,3 mini-GPT,"v3 (ctx=128, layers=4, heads=4)",val_loss,4.7092,2.86,550.60
3,5.2 порівняння моделей,tabularisai/multilingual-sentiment-analysis,accuracy,0.8500,135.00,0.18
4,5.2 порівняння моделей,tabularisai/multilingual-sentiment-analysis,f1,0.8235,135.00,0.18
5,5.2 порівняння моделей,mDeBERTa zero-shot,accuracy,1.0000,279.00,1.99
6,5.2 порівняння моделей,mDeBERTa zero-shot,f1,1.0000,279.00,1.99
7,5.3 налаштування mDeBERTa,база,accuracy,1.0000,NaN,1.18
8,5.3 налаштування mDeBERTa,база,f1,1.0000,NaN,1.18
9,5.3 налаштування mDeBERTa,шаблон гіпотези українською,accuracy,1.0000,NaN,1.03


---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

## 6. Творчий застосунок: «Вгадай слово»

Gradio-гра на основі моделі `xlm-roberta-base` (fill-mask): у реченні українською ховається слово, користувач вводить свою здогадку, а потім застосунок показує топ-5 передбачень моделі з ймовірностями й веде рахунок «користувач : модель». Це дозволяє на практиці побачити, як модель-енкодер використовує контекст з обох боків, і порівняти свою інтуїцію з нею.

In [22]:
!pip install -q gradio

import random
import gradio as gr
from transformers import pipeline

unmasker = pipeline("fill-mask", model="xlm-roberta-base")

SENTENCES = [
    ("Київ — столиця {}.", "України"),
    ("Влітку ми поїхали відпочивати на {}.", "море"),
    ("Взимку на вулиці холодно, і йде {}.", "сніг"),
    ("Щоранку я п'ю гарячу {} перед роботою.", "каву"),
    ("Мій брат навчається в {} у Львові.", "університеті"),
    ("Вчора ми дивилися цікавий {} у кінотеатрі.", "фільм"),
    ("Восени з дерев опадає жовте {}.", "листя"),
    ("Дитина читає цікаву {} перед сном.", "книгу"),
]

with gr.Blocks(title="Вгадай слово") as demo:
    gr.Markdown("## Вгадай слово: ти проти XLM-RoBERTa\n"
                "У реченні сховано слово. Впиши свою здогадку, а потім подивись, що вгадала модель.")
    masked_state, answer_state = gr.State(""), gr.State("")
    score_state = gr.State([0, 0, 0])          # [раунди, ти, модель]
    sentence = gr.Markdown()
    guess = gr.Textbox(label="Твоя відповідь", placeholder="одне слово")
    with gr.Row():
        check_btn = gr.Button("Перевірити", variant="primary")
        new_btn = gr.Button("Нове речення")
    result = gr.Markdown()

    def new_round():
        template, answer = random.choice(SENTENCES)
        return "### " + template.format("_____"), template.format("<mask>"), answer, "", ""

    def check(text, masked, answer, score):
        if not text.strip():
            return "Спочатку впиши слово.", score
        preds = unmasker(masked, top_k=5)
        you = text.strip().lower() == answer.lower()
        model = preds[0]["token_str"].strip().lower() == answer.lower()
        score = [score[0] + 1, score[1] + you, score[2] + model]
        top = "\n".join(f"{i}. **{p['token_str'].strip()}** — {p['score']:.1%}"
                        for i, p in enumerate(preds, 1))
        return (f"Правильна відповідь: **{answer}**\n\n"
                f"{'✅' if you else '❌'} ти · {'✅' if model else '❌'} модель\n\n"
                f"Топ-5 моделі:\n{top}\n\n"
                f"Рахунок після {score[0]} раундів: ти {score[1]} : {score[2]} модель"), score

    outputs_new = [sentence, masked_state, answer_state, guess, result]
    demo.load(new_round, outputs=outputs_new)
    new_btn.click(new_round, outputs=outputs_new)
    check_btn.click(check, inputs=[guess, masked_state, answer_state, score_state],
                    outputs=[result, score_state])

demo.launch()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7b8f2ceb3de8403175.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```